In [1]:
from df import enhance, init_df

from torch_stoi import NegSTOILoss
from torchmetrics.audio.pesq import PerceptualEvaluationSpeechQuality
from torchmetrics.audio import SpeechReverberationModulationEnergyRatio, ShortTimeObjectiveIntelligibility, DeepNoiseSuppressionMeanOpinionScore, ScaleInvariantSignalDistortionRatio
from torchaudio.transforms import Resample

import torch
import torchaudio
import numpy as np
import random

import os

/home/zakhar/miniconda3/envs/ems_dereverb/lib/python3.10/site-packages/df/io.py:9: UserWarning: `torchaudio.backend.common.AudioMetaData` has been moved to `torchaudio.AudioMetaData`. Please update the import path.
  from torchaudio.backend.common import AudioMetaData


In [2]:
SEED = 1984

np.random.seed(SEED)
torch.manual_seed(SEED)
random.seed(SEED)

gen = torch.Generator()
gen.manual_seed(SEED)

np.set_printoptions(precision=3)
torch.set_printoptions(precision=3)

In [3]:
import yaml

from NISQA_s.src.core.model_torch import model_init
from NISQA_s.src.utils.process_utils import process

NISQA_PATH = "NISQA_s/config/nisqa_s.yaml"

with open(NISQA_PATH, 'r') as stream:
    nisqa_args = yaml.safe_load(stream)
nisqa_args["ms_n_fft"] = 512
nisqa_args["hop_length"] = 256
nisqa_args["ms_win_length"] = 512
nisqa_args["ckp"] = nisqa_args["ckp"][3:]

nisqa, h0_nisqa, c0_nisqa = model_init(nisqa_args)

/home/zakhar/miniconda3/envs/ems_dereverb/lib/python3.10/site-packages/torch/nn/modules/rnn.py:83: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=1 and num_layers=1
  warnings.warn("dropout option adds dropout after all but last "


In [4]:
SR = 48_000

# NOISE_PATH = "data/DS_10283_2791/clean_testset_wav"
CLEAN_PATH = "data/DS_10283_2791/clean_testset_wav"
NOISE_PATH = "data/demand_test"
RIRS = {1: os.path.join("data", "rirs48_small_3_test"), 1: os.path.join("data", "rirs48_medium_3_test"), 1: os.path.join("data", "rirs48_large_3_test"), 1: os.path.join("data", "rirs48_super_large_3_test")}
# noise_paths = [os.path.join(NOISE_PATH, x) for x in os.listdir(NOISE_PATH)]
# clean_paths = [os.path.join(CLEAN_PATH, x) for x in os.listdir(CLEAN_PATH)]

# test_data = list(zip(noise_paths, clean_paths))

BATCH_SIZE = 32
DEVICE = "cuda:0"

In [5]:
from src.dataset import *

test_dataset = TRUNetDataset(CLEAN_PATH, sr=SR, noise_dir=NOISE_PATH, rir_dir=RIRS, snr=[0, 5, 10, 15], rir_proba=0.85, noise_proba=0.85, rir_target=False, return_noise=False, return_rir=False)
test_dataset.set_epoch(1)

180
12


In [6]:
tmp1, tmp2, _, _ = test_dataset[4]

In [7]:
from IPython.display import Audio
Audio(tmp1, rate=SR)

In [8]:
srmr = SpeechReverberationModulationEnergyRatio(fs=16_000, norm=False)
stoi = NegSTOILoss(16_000, use_vad=False, do_resample=False).to(DEVICE)
sisdr = ScaleInvariantSignalDistortionRatio().to(DEVICE)
pesq = PerceptualEvaluationSpeechQuality(fs=16_000, mode="wb").to(DEVICE)
dnsmos = DeepNoiseSuppressionMeanOpinionScore(16_000, False, device=DEVICE)

In [9]:
from df import enhance, init_df

model, df_state, _ = init_df()

2026-05-08 13:07:37 | INFO     | DF | Running on torch 2.2.0+cu121
2026-05-08 13:07:37 | INFO     | DF | Running on host zakhar-OMENbyHP
2026-05-08 13:07:37 | INFO     | DF | Loading model settings of DeepFilterNet3
2026-05-08 13:07:37 | INFO     | DF | Using DeepFilterNet3 model at /home/zakhar/.cache/DeepFilterNet/DeepFilterNet3
2026-05-08 13:07:37 | INFO     | DF | Initializing model `deepfilternet3`
2026-05-08 13:07:37 | INFO     | DF | Found checkpoint /home/zakhar/.cache/DeepFilterNet/DeepFilterNet3/checkpoints/model_120.ckpt.best with epoch 120
2026-05-08 13:07:37 | INFO     | DF | Running on device cuda:0
2026-05-08 13:07:37 | INFO     | DF | Model loaded


fatal: not a git repository (or any parent up to mount point /)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).


In [10]:
OUTPUT_PATH = "data/dfn_enhanced/"

In [14]:
from tqdm import tqdm
import time
from scipy.io.wavfile import write

def get_metrics(data, device="cpu"):
    nisqa_scores = []
    pesq_scores = []
    stoi_scores = []
    sisdr_scores = []
    srmr_scores = []
    dns_scores = []

    rtf_full = []
    rtf_chunk = []
    with torch.no_grad():
        for ind, (input_signal, target_signal, _, _) in tqdm(enumerate(data)):
            
            # signal, signal_sr = torchaudio.load(input_path)
            # target, target_sr = torchaudio.load(target_path)

            input_signal = input_signal.to(device)
            target_signal = target_signal.to(device)
            
            # print(signal.shape)
            start_time = time.time()
            output = enhance(model, df_state, input_signal.cpu()).to(device) # torch.from_numpy(enhancer(signal[0].cpu(), signal_sr)).unsqueeze(0).to(device)
            end_time = time.time()

            rtf_full.append((input_signal.shape[-1] / SR) / (end_time - start_time))

            window_size = 48_000 // 4
            for j in range(0, input_signal.shape[-1], window_size):
                chunk = input_signal[..., j:j+window_size]
                
                if chunk.shape[-1] < window_size:
                    continue

                start_time = time.time()
                _ = enhance(model, df_state, chunk.cpu()).to(device)
                end_time = time.time()

                rtf_chunk.append((chunk.shape[-1] / SR) / (end_time - start_time))

            min_l = min(output.shape[-1], target_signal.shape[-1])

            nisqa_score, _, _ = process(output.detach().cpu(), SR, nisqa, h0_nisqa, c0_nisqa, nisqa_args)

            output = output[..., :min_l]
            target_signal = target_signal[..., :min_l]

            write(OUTPUT_PATH + f"{ind}" + "_enhanced.wav", SR, output[0].cpu().detach().numpy())

            resampler = Resample(SR, 16_000)
            output = resampler(output.cpu()).cuda()
            target_signal = resampler(target_signal.cpu()).cuda()

            stoi_score = stoi(output[..., :min_l], target_signal[..., :min_l])
            srmr_score = srmr(output.detach().cpu())
            sisdr_score = sisdr(output[..., :min_l], target_signal[..., :min_l])
            
            min_l = min(output.shape[-1], target_signal.shape[-1])

            srmr_score = srmr(output.detach().cpu())
            dnsmos_score = dnsmos(output.detach())

            try:
                pesq_score = pesq(output[..., :min_l], target_signal[..., :min_l])
            except Exception as e:
                # print(min_l)
                # out_wave_ = output.reshape(-1)
                # target_ = target.reshape(-1)
                # write('exception_out.wav', SR, out_wave_.cpu().detach().numpy())
                # write('exception_in.wav', SR, target_.cpu().detach().numpy())
                continue

            nisqa_scores.append(nisqa_score[0])
            srmr_scores.append(srmr_score)
            stoi_scores.append(stoi_score.cpu())
            sisdr_scores.append(sisdr_score.cpu())
            pesq_scores.append(pesq_score.cpu())
            dns_scores.append(dnsmos_score.cpu())

    nisqa_scores = torch.vstack(nisqa_scores).mean(dim=0)
    stoi_scores = torch.vstack(stoi_scores).mean(dim=0)
    sisdr_scores = torch.vstack(sisdr_scores).mean(dim=0)
    srmr_scores = torch.vstack(srmr_scores).mean(dim=0)
    pesq_scores = torch.vstack(pesq_scores).mean(dim=0)
    dns_scores = torch.vstack(dns_scores).mean(dim=0)

    result = {"nisqa": nisqa_scores, "stoi": stoi_scores, "srmr": srmr_scores, "pesq": pesq_scores, "sisdr": sisdr_score, "dnsmos": dns_scores, 
              "full_rtf": sum(rtf_full) / len(rtf_full), "chunk_rtf": sum(rtf_chunk) / len(rtf_chunk)}
        
    return result

In [15]:
metrics = get_metrics(test_dataset, device=DEVICE)

824it [13:26,  1.02it/s]


In [16]:
print("NISQA score (MOS, NOI, DISC, COL, LOUD):", metrics['nisqa'])
print(f"STOI score: {-metrics['stoi']}")
print(f"SI-SDR score: {metrics['sisdr']}")
print(f"SRMR score: {metrics['srmr']}")
print(f"PESQ-WB score: {metrics['pesq']}")
print(f"DNSMOS score: {metrics['dnsmos']}")
print(f"Full rtf: {metrics['full_rtf']}, {1 / metrics['full_rtf']}")
print(f"Chunk rtf: {metrics['chunk_rtf']}, {1 / metrics['chunk_rtf']}")

NISQA score (MOS, NOI, DISC, COL, LOUD): tensor([4.181, 4.310, 4.063, 4.118, 4.180])
STOI score: tensor([0.903])
SI-SDR score: -10.598501205444336
SRMR score: tensor([9.481])
PESQ-WB score: tensor([2.784])
DNSMOS score: tensor([3.556, 3.354, 4.043, 3.092], dtype=torch.float64)
Full rtf: 98.68603645853318, 0.010133145841967109
Chunk rtf: 36.99367780700344, 0.02703164592655574


In [17]:
from thop import profile
from df.utils import as_complex, as_real, download_file, get_cache_dir, get_norm_alpha
import warnings
from libdf import DF, erb, erb_norm, unit_norm
from df.modules import get_device
from df.model import ModelParams

def df_features(audio, df, nb_df, device=None):
    spec = df.analysis(audio.numpy())  # [C, Tf] -> [C, Tf, F]
    a = get_norm_alpha(False)
    erb_fb = df.erb_widths()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", UserWarning)
        erb_feat = torch.as_tensor(erb_norm(erb(spec, erb_fb), a)).unsqueeze(1)
    spec_feat = as_real(torch.as_tensor(unit_norm(spec[..., :nb_df], a)).unsqueeze(1))
    spec = as_real(torch.as_tensor(spec).unsqueeze(1))
    if device is not None:
        spec = spec.to(device)
        erb_feat = erb_feat.to(device)
        spec_feat = spec_feat.to(device)
    return spec, erb_feat, spec_feat

input_audio_ = torch.ones(1, 960)

bs = input_audio_.shape[0]
if hasattr(model, "reset_h0"):
    model.reset_h0(batch_size=bs, device=get_device())
orig_len = input_audio_.shape[-1]
n_fft, hop = 0, 0

nb_df = getattr(model, "nb_df", getattr(model, "df_bins", ModelParams().nb_df))
spec, erb_feat, spec_feat = df_features(input_audio_, df_state, nb_df, device="cpu")

macs, params = profile(model.cpu(), inputs=(spec, erb_feat, spec_feat))

[INFO] Register count_convNd() for <class 'torch.nn.modules.conv.Conv2d'>.
[INFO] Register count_normalization() for <class 'torch.nn.modules.batchnorm.BatchNorm2d'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.activation.ReLU'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.container.Sequential'>.
[INFO] Register count_gru() for <class 'torch.nn.modules.rnn.GRU'>.
[INFO] Register count_linear() for <class 'torch.nn.modules.linear.Linear'>.
[INFO] Register count_convNd() for <class 'torch.nn.modules.conv.ConvTranspose2d'>.


In [18]:
print("MACs: ", macs)
print("Params: ", params)

MACs:  6589440.0
Params:  2013371.0
